In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path


In [ ]:
PROJECT_DIR = Path.cwd().resolve().parent
TRANSFORM_DIR = PROJECT_DIR / 'processed' / 'transform'

timeseries_cohort = pd.read_csv(TRANSFORM_DIR / 'hourly_timeseries_60min.csv')
assessment_index = pd.read_csv(TRANSFORM_DIR / 'assessment_index_60min.csv')
cohort_final = pd.read_csv(TRANSFORM_DIR / 'cohort_final.csv')

for col in ['intime', 'outtime', 'admittime', 'dischtime', 'deathtime']:
    if col in cohort_final.columns:
        cohort_final[col] = pd.to_datetime(cohort_final[col], errors='coerce')

print(f"Timeseries: {timeseries_cohort.shape}")
print(f"Assessment index: {assessment_index.shape}")
print(f"Cohort stays: {cohort_final['stay_id'].nunique():,}")


## 환자 단위 무작위 train/test 분할

Subject 단위로 80/20 random split을 수행합니다. 같은 subject의 모든 stay와 assessment는 반드시 같은 split에 속합니다.


In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

unique_subjects = np.array(sorted(cohort_final['subject_id'].unique()))
rng = np.random.default_rng(RANDOM_STATE)
shuffled_subjects = rng.permutation(unique_subjects)
n_test = int(np.ceil(len(shuffled_subjects) * TEST_SIZE))
test_subjects = np.sort(shuffled_subjects[:n_test])
train_subjects = np.sort(shuffled_subjects[n_test:])

split_map = {subject_id: 'train' for subject_id in train_subjects}
split_map.update({subject_id: 'test' for subject_id in test_subjects})

timeseries_cohort['split'] = timeseries_cohort['subject_id'].map(split_map)
assessment_index['split'] = assessment_index['subject_id'].map(split_map)
cohort_final['split'] = cohort_final['subject_id'].map(split_map)

train_subject_ids = pd.DataFrame({'subject_id': train_subjects})
test_subject_ids = pd.DataFrame({'subject_id': test_subjects})

print('=== Split overview ===')
print(f"Train subjects: {len(train_subjects):,}")
print(f"Test subjects: {len(test_subjects):,}")
print(f"Overlap subjects: {len(set(train_subjects) & set(test_subjects))}")

subject_split_summary = (
    cohort_final
    .groupby('subject_id')
    .agg(split=('split', 'first'), ever_delirium=('ever_delirium', 'max'))
    .reset_index()
)

print()
print('=== Subject-level ever_delirium by split ===')
display(pd.crosstab(subject_split_summary['split'], subject_split_summary['ever_delirium'], margins=True))

print()
print('=== Stay counts by split ===')
display(cohort_final.groupby('split')['stay_id'].nunique().rename('n_stays').reset_index())

print()
print('=== Assessment counts by split ===')
display(assessment_index.groupby(['split', 'delirium']).size().unstack(fill_value=0))

print()
print('=== Parkinson cohort period by split ===')
display(cohort_final.groupby('split').agg(
    n_subjects=('subject_id', 'nunique'),
    n_stays=('stay_id', 'nunique'),
    first_intime=('intime', 'min'),
    last_outtime=('outtime', 'max'),
))


## Split 산출물 저장


In [ ]:
timeseries_cohort.to_csv(TRANSFORM_DIR / 'hourly_timeseries_60min.csv', index=False)
assessment_index.to_csv(TRANSFORM_DIR / 'assessment_index_60min.csv', index=False)
cohort_final.to_csv(TRANSFORM_DIR / 'cohort_final.csv', index=False)
train_subject_ids.to_csv(TRANSFORM_DIR / 'train_subject_ids.csv', index=False)
test_subject_ids.to_csv(TRANSFORM_DIR / 'test_subject_ids.csv', index=False)

print(f"Saved split column: hourly_timeseries_60min.csv ({len(timeseries_cohort):,} rows)")
print(f"Saved split column: assessment_index_60min.csv ({len(assessment_index):,} rows)")
print(f"Saved split column: cohort_final.csv ({len(cohort_final):,} stays)")
print(f"Saved: train_subject_ids.csv ({len(train_subject_ids):,} subjects)")
print(f"Saved: test_subject_ids.csv ({len(test_subject_ids):,} subjects)")
